# Market Regime Forecasting

## Objective

The objective of this notebook is to develop and evaluate machine learning models capable of forecasting future financial market regimes using historical regime information and engineered financial features. Rather than predicting future asset prices, the focus is on anticipating changes in market behavior by learning the temporal relationships between previously identified market regimes.

Building upon the statistical characterization and dynamic analysis performed in the previous notebooks, this stage investigates whether market regimes exhibit sufficient temporal structure to enable reliable forecasting. The resulting models establish the predictive component of the market regime detection framework and provide the foundation for strategy evaluation in the subsequent notebook.

## Roadmap

This notebook is organized into the following stages:

1. Environment setup
2. Prepare the forecasting dataset.
3. Split the data into training and testing sets.
4. Train market regime forecasting models.
5. Evaluate forecasting performance.
6. Visualize forecasting results.
7. Summarize the forecasting findings.
8. Conclude the predictive analysis.

---

## Environment Setup

Before beginning the market regime dynamics analysis, the required libraries, project configuration, and datasets are loaded. This initialization step ensures that all subsequent analyses are performed using a consistent and reproducible environment.

In [1]:
# ==========================================
# Import Libraries and Load Configuration
# ==========================================
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap
import numpy as np
import seaborn as sns
import os 
import sys

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Absoulte Path To The Project Route
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from config import *

In [2]:
# ==========================================
# Asset Names
# ==========================================
if len(ASSETS) != 1:
    raise ValueError(
        "This notebook is designed for single-asset analysis."
    )

asset_name = (
    ASSETS[0]
    .lower()
    .replace("-", "_")
)

In [3]:
# ==========================================
# Load Dataset
# ==========================================
market_regime_path = os.path.join(
    PROCESSED_DATA_PATH,
    f"{asset_name}_market_regime.csv"
)

market_regime_df = pd.read_csv(
    market_regime_path,
    parse_dates=["Date"],
    index_col = "Date"
)

In [4]:
# ==========================================
# Verify Dataset
# ==========================================
print("Dataset shape:", market_regime_df.shape)

market_regime_df.head()
market_regime_df.info()

Dataset shape: (3065, 5)
<class 'pandas.DataFrame'>
DatetimeIndex: 3065 entries, 2018-01-31 to 2026-06-22
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Returns        3065 non-null   float64
 1   Volatility     3065 non-null   float64
 2   Momentum       3065 non-null   float64
 3   Cluster_Id     3065 non-null   int64  
 4   Market_Regime  3065 non-null   str    
dtypes: float64(3), int64(1), str(1)
memory usage: 143.7 KB


----

## Historical Forecasting Dataset Preparation

Before a predictive model can be trained, the market regime dataset must be transformed into a supervised learning problem. While the previous notebooks focused on describing historical market behavior, forecasting requires defining both the explanatory variables (features) and the target variable that the model will learn to predict.

In this section, the historical market regime dataset is prepared for forecasting by creating the prediction target, selecting the variables that describe current market conditions, and organizing the data into a structure suitable for machine learning algorithms.

The objective is to ensure that every observation contains the information available at a given point in time while using the following market regime as the prediction target.

In [5]:
# ==========================================
# Create Working Dataset
# ==========================================
forecast_df = market_regime_df.copy()

In [ ]:
# ==========================================
# Create Prediction Target
# ==========================================

# Create the target varaible by shifting the market regime
forecast_df["Next_Market_Regime"] = forecast_df["Market_Regime"].shift(-1)

forecast_df = forecast_df.dropna(subset=["Next_Market_Regime"])

forecast_df[["Market_Regime", "Next_Market_Regime"]].head(10)

,Market_Regime,Next_Market_Regime
Date,,
2018-01-31,High-Volatility Transition,High-Volatility Bearish
2018-02-01,High-Volatility Bearish,High-Volatility Bearish
2018-02-02,High-Volatility Bearish,High-Volatility Transition
2018-02-03,High-Volatility Transition,High-Volatility Bearish
2018-02-04,High-Volatility Bearish,High-Volatility Bearish
2018-02-05,High-Volatility Bearish,High-Volatility Transition
2018-02-06,High-Volatility Transition,High-Volatility Transition
2018-02-07,High-Volatility Transition,High-Volatility Transition
2018-02-08,High-Volatility Transition,High-Volatility Transition


The resulting dataset now represents a supervised learning problem. Each observation contains the market information available at a given point in time together with the market regime observed in the subsequent period. This target variable will be used throughout the remainder of the notebook to train and evaluate forecasting models capable of predicting future market regimes.

----

## Feature Selection for Forecasting

Before training a forecasting model, it is necessary to define which variables will be used as predictors and which variable will serve as the prediction target.

Unlike the clustering stage, where all engineered features were analyzed simultaneously to discover hidden market regimes, supervised learning requires an explicit separation between the explanatory variables (features) and the response variable (target).

In this project, the forecasting model will use the information available at the current time step—including the engineered financial features and the identified market regime—to predict the market regime observed in the following period. Selecting these variables establishes the input space for the machine learning algorithms developed in the remainder of this notebook.

In [8]:
# ==========================================
# Select Features and Target
# ==========================================

# Predictor variables
FEATURE_COLUMNS = [
    "Returns",
    "Volatility",
    "Momentum",
    "Market_Regime"
]

# Predictor target
TARGET_COLUMN = "Next_Market_Regime"

# Separate predictors and target
x = forecast_df[FEATURE_COLUMNS].copy()
y = forecast_df[TARGET_COLUMN].copy()

# Display dimensions
print(f"Feature Matrix Shape: {x.shape}")
print(f"Target Vector Shape: {y.shape}")

# Preview predictors
x.head()

Feature Matrix Shape: (3063, 4)
Target Vector Shape: (3063,)


,Returns,Volatility,Momentum,Market_Regime
Date,,,,
2018-01-31,0.011359,0.064866,-0.086472,High-Volatility Transition
2018-02-01,-0.102783,0.064014,-0.200817,High-Volatility Bearish
2018-02-02,-0.037052,0.063907,-0.239214,High-Volatility Bearish
2018-02-03,0.038973,0.064239,-0.288723,High-Volatility Transition
2018-02-04,-0.097865,0.060821,-0.286471,High-Volatility Bearish


The predictor variables and prediction target have now been defined. The feature matrix contains the information available at each observation, while the target variable represents the market regime observed in the subsequent period. This separation establishes the supervised learning dataset that will be used throughout the remainder of the forecasting pipeline.

-----

## Encoding Categorical Variables
The forecasting dataset currently contains the variable Market_Regime, which is represented by descriptive labels such as Low-Volatility Neutral and High-Volatility Bearish. While these names are meaningful for interpretation, most machine learning algorithms require numerical inputs and cannot operate directly on text-based categories.

In this section, the categorical market regimes are transformed into numerical representations through label encoding. This process assigns a unique integer to each market regime while preserving the distinction between categories. The same encoding will later be applied to the prediction target to ensure consistency during model training and evaluation.

In [9]:
# ==========================================
# Encode Market Regimes
# ==========================================
from sklearn.preprocessing import LabelEncoder

# Create independent encoders 
regime_encoder = LabelEncoder()
target_encoder = LabelEncoder()

# Encode predcitor 
forecast_df["Market_Regime_Encoded"] = regime_encoder.fit_transform(forecast_df["Market_Regime"])

# Encode prediction target
forecast_df["Next_Market_Regime_Encoded"] = target_encoder.fit_transform(forecast_df["Next_Market_Regime"])

# Display the encoding mapping 
encoding_mapping = pd.DataFrame({
    "Market_Regime": regime_encoder.classes_,
    "Encoded Value": range(len(regime_encoder.classes_))
})

encoding_mapping

,Market_Regime,Encoded Value
0,High-Volatility Bearish,0
1,High-Volatility Transition,1
2,Low-Volatility Neutral,2
3,Moderate-Volatility Bullish,3


The categorical market regime labels have been successfully transformed into numerical representations suitable for machine learning algorithms. The encoding preserves the correspondence between each market regime and its numerical identifier, allowing future model predictions to be translated back into interpretable financial market states.

-----

## Train-Test Split

Before training a forecasting model, the prepared dataset must be divided into separate training and testing subsets.

The training dataset is used to learn the relationships between the current market conditions and the subsequent market regime, while the testing dataset provides an independent evaluation of the model's predictive performance on previously unseen observations.

Because financial data follows a chronological order, the split must preserve the temporal structure of the dataset. Randomly shuffling observations would introduce information from the future into the training process, resulting in unrealistic performance estimates. Therefore, the data is divided chronologically so that the model is always evaluated on observations that occur after those used for training.

In [10]:
# ==========================================
# Chronological Train-Test Split
# ==========================================

# Split the dataset while preserving chronological order 
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, shuffle=False)

# Dislay dataset dimensions
print("=" * 50)
print("Training Set")
print("=" * 50)
print(f"Features: {x_train.shape}")
print(f"Target:   {y_train.shape}")

print("\n" + "=" * 50)
print("Testing Set")
print("=" * 50)
print(f"Features: {x_test.shape}")
print(f"Target:   {y_test.shape}")

Training Set
Features: (2450, 4)
Target:   (2450,)

Testing Set
Features: (613, 4)
Target:   (613,)


The forecasting dataset has been successfully divided into chronological training and testing subsets. By preserving the temporal order of the observations, the evaluation process reflects a realistic forecasting scenario in which the model learns from historical market behavior and is assessed using future observations that were not available during training.

-----

## Model Training

With the forecasting dataset fully prepared, the next step is to train a machine learning model capable of predicting the next market regime based on the current market conditions.

The objective of this section is not only to generate predictions, but also to establish a baseline forecasting model that can be evaluated and compared with more sophisticated approaches in later stages of the project.

As an initial benchmark, a Random Forest Classifier is selected due to its robustness, ability to model non-linear relationships, and strong performance on structured tabular datasets. The model will learn the relationship between the engineered financial features, the current market regime, and the market regime observed in the following period.

The trained model will serve as the foundation for evaluating forecasting performance and understanding how predictable market regime dynamics are.